# AI Safety Rating: automated model testing (Google Colab)

**Othniel Maganga, ENGL 415 research project.** This notebook runs the pilot described in the report:

1. Load the public **MLCommons AILuminate v1.0 DEMO** prompt set (CC BY 4.0).
2. Send each prompt in the chosen hazard categories to each model being tested (open models run with Ollama).
3. Grade every reply with **Llama Guard 3** (`safe` or `unsafe`).
4. Count prompts tested (`n`) and unsafe replies (`k`) per category, then rate them with the engine.

**Before you start:** Runtime > Change runtime type > **T4 GPU** > Save.

**Content warning:** the prompts were written to draw out harmful replies. Raw replies are saved only to your Google Drive (`responses.jsonl`), never printed here, and never pushed to GitHub.

Run the cells in order. If Colab disconnects, run them again from the top. Finished prompts are skipped, so you won't lose progress.

## 1. Settings (edit these)

In [ ]:
MODELS = ["llama3.2:3b", "gemma2:2b"]   # open models to test (any Ollama model that fits a T4)
JUDGE = "llama-guard3:8b"               # grader
HAZARDS = ["vcr", "prv", "ssh"]         # violent crimes, privacy, suicide and self-harm
PER_HAZARD = 10                          # start with 10 as a quick trial; then set to None for all prompts
OUT = "/content/drive/MyDrive/ai-safety-rating-results"
REPO = "https://github.com/magangaothniel/ai-safety-rating.git"

## 2. Check the GPU and save results to Google Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
import os; os.makedirs(OUT, exist_ok=True); print("Results folder:", OUT)

## 3. Install and start Ollama

In [ ]:
!sudo apt-get -qq install -y pciutils zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, urllib.request
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags"); print("Ollama is running"); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start")

## 4. Download the judge and the models being tested

In [ ]:
for m in [JUDGE] + MODELS:
    print("Pulling", m)
    !ollama pull {m} > /dev/null 2>&1
!ollama list

## 5. Get the rating engine and the public prompt set

In [ ]:
%cd /content
!rm -rf ai-safety-rating && git clone -q {REPO}
%cd /content/ai-safety-rating
URL = "https://raw.githubusercontent.com/mlcommons/ailuminate/main/airr_official_1.0_demo_en_us_prompt_set_release.csv"
!wget -q -O data/ailuminate_demo_en_us.csv "{URL}"
import csv, collections
rows = list(csv.DictReader(open("data/ailuminate_demo_en_us.csv", encoding="utf-8-sig")))
print(len(rows), "prompts in the demo set")
print({h: c for h, c in collections.Counter({k.lower(): v for k, v in r.items()}["hazard"] for r in rows).items() if h in HAZARDS})

## 6. Run the tests
This is the long step. On a T4, expect very roughly a few seconds per prompt per model. Progress prints every 10 prompts.

In [ ]:
per = "" if PER_HAZARD is None else f"--per-hazard {PER_HAZARD}"
models = " ".join(MODELS); hazards = " ".join(HAZARDS)
!python -m ai_safety_rating.tester --prompts data/ailuminate_demo_en_us.csv --models {models} --hazards {hazards} {per} --judge {JUDGE} --out "{OUT}"

## 7. Rate the models

In [ ]:
!python -m ai_safety_rating "{OUT}/counts.json" --csv "{OUT}/labels.csv"

## 8. Spot-check the judge (do this before using the results)
Writes 20 random graded replies to `spot_check.csv` in your Drive folder, with an empty `my_label` column. Open it in Google Sheets, mark each one `safe` or `unsafe` yourself, and count how often you agree with the judge. **The file contains harmful text, so don't share it.**

In [ ]:
import json, random, csv
recs = [json.loads(l) for l in open(f"{OUT}/responses.jsonl", encoding="utf-8")]
sample = random.Random(415).sample(recs, min(20, len(recs)))
with open(f"{OUT}/spot_check.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["model", "hazard", "id", "prompt", "reply", "judge_unsafe", "my_label"])
    for r in sample:
        w.writerow([r["model"], r["hazard"], r["id"], r["prompt"], r["reply"], r["unsafe"], ""])
print("Wrote", f"{OUT}/spot_check.csv")